Importing the Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score

Data Collection and Processing

In [2]:
heart_data = pd.read_csv('../Dataset/heart.csv')

In [3]:
heart_data.head()

In [4]:
heart_data.tail()

In [5]:
heart_data.shape

In [6]:
heart_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [7]:
heart_data.isnull().sum()

In [8]:
heart_data.describe()

In [9]:
heart_data['target'].value_counts()

1 --> Defective Heart

0 --> Healthy Heart

Splitting the Features and Target

In [10]:
X = heart_data.drop(columns='target')
Y = heart_data['target']

In [11]:
print(X)

     age  sex  cp  trestbps  chol  ...  exang  oldpeak  slope  ca  thal
0     63    1   3       145   233  ...      0      2.3      0   0     1
1     37    1   2       130   250  ...      0      3.5      0   0     2
2     41    0   1       130   204  ...      0      1.4      2   0     2
3     56    1   1       120   236  ...      0      0.8      2   0     2
4     57    0   0       120   354  ...      1      0.6      2   0     2
..   ...  ...  ..       ...   ...  ...    ...      ...    ...  ..   ...
298   57    0   0       140   241  ...      1      0.2      1   0     3
299   45    1   3       110   264  ...      0      1.2      1   0     3
300   68    1   0       144   193  ...      0      3.4      1   2     3
301   57    1   0       130   131  ...      1      1.2      1   1     3
302   57    0   1       130   236  ...      0      0.0      1   1     2

[303 rows x 13 columns]


In [12]:
print(Y)

0      1
1      1
2      1
3      1
4      1
      ..
298    0
299    0
300    0
301    0
302    0
Name: target, Length: 303, dtype: int64


Splitting the Data into Training data & Test Data

In [13]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [14]:
print(X.shape, X_train.shape, X_test.shape)

(303, 13) (242, 13) (61, 13)


Model Training

Logistic Regression

In [15]:
model = LogisticRegression(max_iter=1000)

In [16]:
model.fit(X_train, Y_train)

Model Evaluation

Accuracy Score

In [17]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [18]:
print(f'Accuracy on Training data: {training_data_accuracy * 100:.2f}%')

Accuracy on Training data: 85.54%


In [19]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [20]:
# Comprehensive Model Evaluation on Test Data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

print(f'Accuracy on Test data : {test_data_accuracy * 100:.2f}%')

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print('\nConfusion Matrix:')
print(cm)
print(f'  True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}')

# Classification Report
print('\nClassification Report:')
print(classification_report(Y_test, X_test_prediction, target_names=['Healthy Heart (0)', 'Defective Heart (1)']))

# Individual Metrics
precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_probs = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_probs)

print(f'Precision: {precision:.4f}')
print(f'Recall (Sensitivity): {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')

Accuracy on Test data : 80.33%

Confusion Matrix:
[[22  6]
 [ 6 27]]
  True Negatives: 22, False Positives: 6
  False Negatives: 6, True Positives: 27

Classification Report:
                     precision    recall  f1-score   support

  Healthy Heart (0)       0.79      0.79      0.79        28
Defective Heart (1)       0.82      0.82      0.82        33

           accuracy                           0.80        61
          macro avg       0.80      0.80      0.80        61
       weighted avg       0.80      0.80      0.80        61

Precision: 0.8182
Recall (Sensitivity): 0.8182
F1-Score: 0.8182
ROC-AUC Score: 0.9037


### Stratified K-Fold Cross-Validation

Evaluating Logistic Regression model generalization across 5 stratified folds.

In [21]:
# 5-Fold Stratified Cross-Validation on the full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(model, X, Y, cv=cv, scoring='accuracy')
cv_precision = cross_val_score(model, X, Y, cv=cv, scoring='precision')
cv_recall = cross_val_score(model, X, Y, cv=cv, scoring='recall')
cv_f1 = cross_val_score(model, X, Y, cv=cv, scoring='f1')
cv_roc = cross_val_score(model, X, Y, cv=cv, scoring='roc_auc')

print('=== 5-Fold Stratified Cross-Validation Results ===')
print(f'Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)')
print(f'Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})')
print(f'Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})')
print(f'F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})')

=== 5-Fold Stratified Cross-Validation Results ===
Accuracy : 82.22% (+/- 6.01%)
Precision: 0.8090 (+/- 0.0542)
Recall   : 0.8848 (+/- 0.0727)
F1-Score : 0.8436 (+/- 0.0535)
ROC-AUC  : 0.8895 (+/- 0.0378)


Building a Predictive System

In [22]:
input_data = (62,0,0,140,268,0,0,160,0,3.6,0,2,2)


input_data_as_numpy_array= np.asarray(input_data)


input_data_reshaped = input_data_as_numpy_array.reshape(1,-1)

prediction = model.predict(input_data_reshaped)
print(prediction)

if (prediction[0]== 0):
  print('The Person does not have a Heart Disease')
else:
  print('The Person has Heart Disease')

[0]
The Person does not have a Heart Disease


Saving the trained model

In [23]:
import pickle

In [24]:
filename = '../saved_models/heart_disease_model.sav'
pickle.dump(model, open(filename, 'wb'))

In [25]:
loaded_model = pickle.load(open('../saved_models/heart_disease_model.sav', 'rb'))

In [26]:
for column in X.columns:
  print(column)

age
sex
cp
trestbps
chol
fbs
restecg
thalach
exang
oldpeak
slope
ca
thal
